In [ ]:
#!/usr/bin/env python3
"""Simple H5AD clustering tool with K-means, Bisecting K-means, and Leiden."""

import numpy as np
import warnings
warnings.filterwarnings('ignore')

import anndata as ad
import scanpy as sc
import pandas as pd
from sklearn.cluster import KMeans, BisectingKMeans
import matplotlib.pyplot as plt



In [ ]:

def load_h5ad(filepath):
    """Load h5ad file and preprocess if needed."""
    print(f"Loading {filepath}...")
    adata = sc.read_h5ad(filepath)
    print(f"  {adata.n_obs} cells × {adata.n_vars} genes")
    
    # Preprocess if PCA not present
    if 'X_pca' not in adata.obsm:
        print("  Preprocessing...")
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        sc.pp.scale(adata, max_value=10)
        sc.tl.pca(adata, n_comps=50)
    
    return adata


def cluster_kmeans(adata, n_clusters=10):
    """K-means clustering."""
    print(f"Running K-means (k={n_clusters})...")
    data = adata.obsm['X_pca']
    labels = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit_predict(data)
    adata.obs['cluster'] = pd.Categorical([f"C{i}" for i in labels])
    return adata


def cluster_bisecting_kmeans(adata, n_clusters=10):
    """Bisecting K-means clustering."""
    print(f"Running Bisecting K-means (k={n_clusters})...")
    data = adata.obsm['X_pca']
    labels = BisectingKMeans(n_clusters=n_clusters, random_state=42).fit_predict(data)
    adata.obs['cluster'] = pd.Categorical([f"C{i}" for i in labels])
    return adata


def cluster_leiden(adata, resolution=1.0):
    """Leiden clustering."""
    print(f"Running Leiden (resolution={resolution})...")
    if 'neighbors' not in adata.uns:
        sc.pp.neighbors(adata, n_neighbors=15)
    sc.tl.leiden(adata, resolution=resolution, random_state=42)
    adata.obs['cluster'] = adata.obs['leiden']
    return adata


def plot_clusters(adata, save_path=None):
    """Plot clusters on UMAP."""
    if 'X_umap' not in adata.obsm:
        print("Computing UMAP...")
        if 'neighbors' not in adata.uns:
            sc.pp.neighbors(adata, n_neighbors=15)
        sc.tl.umap(adata)
    
    print("Plotting...")
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(adata, color='cluster', ax=ax, show=False, legend_loc='on data')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    
    plt.show()
    return fig


def show_clusters(adata):
    """Print cluster counts."""
    counts = adata.obs['cluster'].value_counts().sort_index()
    print(f"\nCluster counts ({len(counts)} clusters):")
    for name, count in counts.items():
        print(f"  {name}: {count} cells ({100*count/len(adata):.1f}%)")



In [ ]:

# Example usage
if __name__ == "__main__":
  
    
    # Load and cluster
    adata = load_h5ad("demo.h5ad")
    
    # Choose one:
    # adata = cluster_kmeans(adata, n_clusters=5)
    # adata = cluster_bisecting_kmeans(adata, n_clusters=5)
    adata = cluster_leiden(adata, resolution=0.5)
    
    show_clusters(adata)
    plot_clusters(adata, save_path="clusters.png")
    
    # Save results
    adata.write_h5ad("demo_clustered.h5ad")
    print("Done!")